# Task 1.1 — Core Contribution / Architecture (8 marks)

**Paper:** *Online Discovery and Maintenance of Time Series Motifs*  
Mueen, A., Keogh, E., Zhu, Q., Cash, S., Westover, B. — **KDD 2010**

---

## Step-by-Step Method Description

---

### Step 1: Z-normalise Every Incoming Subsequence

**Description:** As the stream advances by one time step, a new subsequence of length *m* is extracted from the sliding window W. Before any comparison, it is z-normalised (zero mean, unit standard deviation). The normalised version is treated as an independent point in ℝᵐ, completely decoupled from its position in the stream for distance purposes.

**Reference:** Section 2 (Notation and Background); also the paragraph immediately after Definition 5 that explicitly states z-normalisation makes subsequences "independent objects" in a high-dimensional space.

**Purpose:** Ensures the algorithm is invariant to baseline shifts and amplitude scaling, so that two occurrences of the same shape (e.g., a heartbeat at different voltages) are correctly identified as similar.

---

### Step 2: Initialise per-Point N-lists and R-lists

**Description:** Every active point (z-normalised subsequence currently inside the sliding window) maintains:
- An **N-list** (Neighbour List): time-stamped list of other points it is nearest to, with entries ordered in strictly increasing timestamp order **and** satisfying a monotonically non-increasing distance property (imposed by Observation 2, Section 4.2). Any entry *y* is pruned if a later-arriving point *z* satisfies *d(x,z) < d(x,y)*.
- An **R-list** (Reverse List): the set of later points for which this point is currently the nearest earlier point — i.e., points that hold this point at the top of their N-list.

**Reference:** Section 4.1 (linked-list formulation), Figure 4 (data structure visualisation), Observation 1 and Observation 2 in Section 4.2.

**Purpose:** The N-list + R-list pair is the core data structure that enables O(w) amortised motif maintenance without recomputing all O(w²) pairwise distances on every window slide.

---

### Step 3: Insertion — Build the New Point's N-list and Update R-lists

**Description:** When a new point *p* arrives:
1. Compute the Euclidean distance from *p* to all *w* existing points — O(wm) work.
2. Sort these by distance, then apply Observation 2: walk through in strictly increasing timestamp order and keep only entries where d(p, ·) is non-increasing; discard the rest. This forms p's N-list.
3. For the head of p's N-list (i.e., p's nearest earlier neighbour *x*), add *p* to x's R-list (per Observation 1 — each pair is stored only once).
4. For any older point *q* whose nearest neighbour has now changed to *p* (discovered while computing distances), update q's R-list to point to p — but do **not** rebuild q's N-list; the old entry remains and may be lazily cleaned on deletion.

**Reference:** Section 4.1 (update upon insertion), Observation 1 and Figure 4(b).

**Purpose:** Keeps each N-list sparse (at most O(√w) entries in the amortised case) by discarding provably non-motif candidates immediately.

---

### Step 4: Deletion — Remove the Oldest Point via R-lists

**Description:** When the oldest point *q* departs the sliding window:
1. Consult q's R-list to find all points *x* that had *q* at the head of their N-list.
2. For each such *x*, delete the head of x's N-list and promote the next valid entry to the head. If the next entry has already expired (timestamp < current window start), delete it too.
3. Update R-lists accordingly: *x* is inserted into the R-list of its new nearest earlier neighbour.

**Reference:** Section 4.2 (update upon deletion), Figure 4 (b) and the paragraph on amortised O(w) deletion.

**Purpose:** Avoids recomputing distances on deletion by reusing the precomputed, still-valid N-list entries. The amortised cost is O(w) because each distance is computed once (on insertion) and removed at most once (on deletion).

---

### Step 5: Answer the Motif Query in O(w) Time

**Description:** At any point in the stream, the current online motif pair is found by scanning all N-list heads and returning the pair with the smallest stored distance. Because the N-lists are guaranteed to contain the closest valid earlier neighbour for each point, the global minimum is guaranteed to be among these heads.

**Reference:** Section 4.2, last paragraph: "the motif pair is guaranteed to be among the first points of the N-lists."

**Purpose:** This allows O(w) motif lookup after every window slide — the primary correctness guarantee of the algorithm.

---

## Final Summary Sentence

This paper solves the problem of **continuously maintaining the closest pair of non-overlapping z-normalised subsequences (the time series motif) in an infinite data stream under a sliding window model**, and the authors claim their O(wm) insertion / O(w) amortised deletion update algorithm is the first **practical exact** approach — superseding the naive O(w²m) streaming baseline by exploiting two structural pruning observations (Observation 1: store each pair once; Observation 2: timestamp–distance monotone N-lists) that dramatically reduce the number of distance computations and the space required per window slide.


In [1]:
# No code required for Task 1.1 — all answers are in markdown cells above.
print("Task 1.1: Paper architecture description complete. All responses in markdown cells.")


Task 1.1: Paper architecture description complete. All responses in markdown cells.
